# 🖼️ Image Caption Generator — train on Google Colab (free GPU)

**First:** `Runtime → Change runtime type → T4 GPU → Save`, then `Runtime → Run all`.

Total time ≈ 30–45 min (feature extraction ~5 min, training ~15–30 min, evaluation ~5 min).
At the end `model_files.zip` downloads. Put its 4 files into the `model/` folder of your GitHub repo.

## 1. Check GPU and versions

In [ ]:
import tensorflow as tf, keras
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "NONE - switch runtime to T4 GPU")

## 2. Get the project code
Change `REPO_URL` if your username or repo name is different.

In [ ]:
REPO_URL = "https://github.com/Anuanshika/Image-Caption-Generator.git"

import os
if not os.path.exists("image-caption-generator"):
    !git clone -q $REPO_URL image-caption-generator
%cd image-caption-generator
!ls

## 3. Install extras
Colab already has TensorFlow and matplotlib.

In [ ]:
!pip install -q kagglehub nltk

## 4. (Only if the download fails) Log in to Kaggle
Flickr8k is public, so this is usually not needed. If step 5 says *Kaggle download failed*, uncomment, run, paste your Kaggle username + API key (kaggle.com → Settings → API → Create New Token), then re-run step 5.

In [ ]:
# import kagglehub
# kagglehub.login()

## 5. Download Flickr8k, extract CNN features, train the Transformer, evaluate

In [ ]:
!python train.py --epochs 20

## 6. Results

In [ ]:
import json
from IPython.display import Image, display
display(Image("reports/training_curves.png"))
print(json.dumps(json.load(open("reports/bleu_scores.json")), indent=2))
display(Image("reports/examples.png"))

## 7. Try it on any image URL

In [ ]:
import io, urllib.request
from PIL import Image as PILImage
from captioner import Captioner

url = "https://images.unsplash.com/photo-1517849845537-4d257902454a?w=640"  # replace with any image URL
try:
    img = PILImage.open(io.BytesIO(urllib.request.urlopen(url).read()))
except Exception as e:
    raise SystemExit(f"Could not load the image ({e}). Paste a different image URL above.")
cap = Captioner()
result, _ = cap.caption(img, method="beam", beam_width=3)
display(img.resize((320, int(320 * img.height / img.width))))
print("Caption:", result.caption)

## 8. Download the trained model

In [ ]:
!zip -j -q model_files.zip model/encoder.weights.h5 model/decoder.weights.h5 model/config.json model/vocab.json
!zip -j -q report_files.zip reports/*.png reports/*.json
!ls -lh model/
from google.colab import files
files.download("model_files.zip")
files.download("report_files.zip")

## 9. Next steps
1. Unzip `model_files.zip` → upload the 4 files to `model/` in your GitHub repo (**Add file → Upload files → Commit**).
2. Unzip `report_files.zip` → upload to `reports/` so the README shows your real results.
3. Deploy on Streamlit Community Cloud (see README).